<a href="https://colab.research.google.com/github/RenteriaRaul/Doctorado-DCAG/blob/main/notebooks/sustax/01_sustax_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Configuración e imports**

In [ ]:
from google.colab import drive
import os
import glob
import re
from pathlib import Path

import numpy as np
import pandas as pd

drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Doctorado /Probabilidad/Precipitación"
RAW_DIR = os.path.join(PROJECT_DIR, "Datos Sustax")
OUT_DIR = os.path.join(RAW_DIR, "SUSTAX_TOTAL_MERGED")

os.makedirs(OUT_DIR, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_DIR:", RAW_DIR)
print("OUT_DIR:", OUT_DIR)

# **Loader seguro**

In [ ]:
def load_sustax_file_safe(csv_stx, return_pandas_df=True, return_metadata=False):
    """
    Wrapper seguro para archivos Sustax con fechas tipo M/D/YYYY.
    """
    import csv
    import decimal
    import numpy as np
    import pandas as pd

    def _isfloat(string):
        try:
            decimal.Decimal(string)
            return True
        except decimal.InvalidOperation:
            return False

    with open(csv_stx, 'r', encoding=None) as fobj:
        data = list(csv.reader(fobj, delimiter=','))

    dts_type = [d for d in data if (len(d) > 0) and (d[0] == 'Dataset:')][0][1]
    idx_data = [c for c in range(len(data)) if 'Data requested:' in data[c]][0] + 4
    all_data = data[idx_data:]
    all_data_vars = data[idx_data-3]
    all_data_scenarios = data[idx_data-2]

    mt = {}
    if return_metadata:
        for r in data:
            if any(['longitude' in h.lower() for h in r]):
                mt["lon"] = [float(rr) for rr in r if _isfloat(rr)][0]
            if any(['latitude' in h.lower() for h in r]):
                mt["lat"] = [float(rr) for rr in r if _isfloat(rr)][0]

    dt = {}
    for c in range(len(all_data_vars)):
        if all_data_vars[c] != '':
            dt.setdefault(all_data_vars[c], {}).update({all_data_scenarios[c]: []})

    time = []
    for r in all_data:
        time.append(pd.to_datetime(r[0], format="%m/%d/%Y"))
        for c in range(len(r)):
            if ("SSP" in all_data_scenarios[c]) or ("ERA" in all_data_scenarios[c]):
                try:
                    dt[all_data_vars[c]][all_data_scenarios[c]].append(
                        float(r[c]) if r[c] != '' else np.nan
                    )
                except:
                    pass

    time = np.array(time, dtype='datetime64[D]')
    dt = {k: {s: np.asarray(dt[k][s]) for s in dt[k]} for k in dt}

    if return_pandas_df:
        dfs = [
            pd.Series({d: v for d, v in zip(time, dt[var][s])}).to_frame(name=f"{var} [{s}]")
            for var in dt for s in dt[var]
        ]
        df_vals = dfs[0].join(dfs[1:])
        df_metrics = pd.DataFrame()

        if return_metadata:
            return df_vals, df_metrics, mt
        else:
            return df_vals, df_metrics
    else:
        if return_metadata:
            return dt, {}, time, mt
        else:
            return dt, {}, time

# **Inventario de archivos Sustax**

In [ ]:
def normalize_sustax_base_name(filename: str) -> str:
    stem = Path(filename).stem
    stem = re.sub(r"(ERA5|SSP119|SSP126|SSP245|SSP343|SSP370|SSP434|SSP460|SSP585)$", "", stem)
    stem = re.sub(r"\d+$", "", stem)
    return stem

def is_real_sustax_csv(fp: str) -> bool:
    name = os.path.basename(fp).lower()

    if not name.endswith(".csv"):
        return False

    ignore_patterns = [
        "catalog",
        "neighbors",
        "summary",
        "obs_",
        "amax",
        "compare",
        "merged",
        "exported"
    ]

    if any(p in name for p in ignore_patterns):
        return False

    if not name.startswith("sustax_"):
        return False

    return True

rows = []
bad = []

csv_files = sorted([fp for fp in glob.glob(os.path.join(RAW_DIR, "*.csv")) if is_real_sustax_csv(fp)])

print("CSV Sustax originales detectados:", len(csv_files))

for fp in csv_files:
    fn = os.path.basename(fp)

    try:
        df_vals, df_metrics, meta = load_sustax_file_safe(
            fp,
            return_pandas_df=True,
            return_metadata=True
        )

        rows.append({
            "sustax_file": fn,
            "base_name": normalize_sustax_base_name(fn),
            "lat": round(float(meta.get("lat", np.nan)), 4),
            "lon": round(float(meta.get("lon", np.nan)), 4),
            "n_rows": len(df_vals),
            "n_cols": df_vals.shape[1],
            "scenarios_detected": ", ".join(sorted({
                c.split("[")[-1].replace("]", "").strip()
                for c in df_vals.columns if "[" in c and "]" in c
            })),
            "path": fp
        })

    except Exception as e:
        bad.append({
            "file": fn,
            "error": str(e)
        })

inventory_df = pd.DataFrame(rows).sort_values(["base_name", "lat", "lon", "sustax_file"])
bad_df = pd.DataFrame(bad)

display(inventory_df.head())

if not bad_df.empty:
    print("Archivos con error:")
    display(bad_df)

group_summary = (
    inventory_df
    .groupby(["base_name", "lat", "lon"], as_index=False)
    .agg(
        n_files=("sustax_file", "count"),
        files=("sustax_file", lambda x: " | ".join(x))
    )
    .sort_values(["base_name", "lat", "lon"])
)

display(group_summary.head())

inventory_path = os.path.join(OUT_DIR, "SUSTAX_raw_inventory.csv")
group_summary_path = os.path.join(OUT_DIR, "SUSTAX_group_summary.csv")

inventory_df.to_csv(inventory_path, index=False)
group_summary.to_csv(group_summary_path, index=False)

print("Inventario guardado en:", inventory_path)
print("Resumen de grupos guardado en:", group_summary_path)

# **Merge LONG y generación por escenario**

In [ ]:
MERGED_LONG_DIR = os.path.join(OUT_DIR, "LONG_TOTAL")
BYSCEN_TOTAL_DIR = os.path.join(OUT_DIR, "BY_SCENARIO_TOTAL")

os.makedirs(MERGED_LONG_DIR, exist_ok=True)
os.makedirs(BYSCEN_TOTAL_DIR, exist_ok=True)

def split_var_scenario(colname: str):
    m = re.match(r"^(.*)\s+\[(.*)\]\s*$", str(colname).strip())
    if not m:
        return None, None
    return m.group(1).strip(), m.group(2).strip()

merged_rows = []

for (base_name, lat, lon), grp in inventory_df.groupby(["base_name", "lat", "lon"]):
    long_parts = []

    for _, row in grp.iterrows():
        fp = row["path"]
        fn = row["sustax_file"]

        try:
            df_vals, df_metrics, meta = load_sustax_file_safe(
                fp,
                return_pandas_df=True,
                return_metadata=True
            )

            df_vals = df_vals.copy()
            df_vals.index = pd.to_datetime(df_vals.index, errors="coerce")
            df_vals = df_vals[~df_vals.index.isna()].sort_index()
            df_vals.index = df_vals.index.normalize()

            iso_dates = df_vals.index.strftime("%Y-%m-%d")

            rows_long = []
            for c in df_vals.columns:
                var, sc = split_var_scenario(c)
                if var is None:
                    continue

                values_mm = pd.to_numeric(df_vals[c], errors="coerce") * 1000.0

                tmp = pd.DataFrame({
                    "source_file": fn,
                    "date": iso_dates,
                    "variable": var,
                    "scenario": sc,
                    "value_mm": values_mm.values
                })
                rows_long.append(tmp)

            if rows_long:
                part = pd.concat(rows_long, ignore_index=True)
                long_parts.append(part)

        except Exception as e:
            print(f"Error en {fn}: {e}")

    if not long_parts:
        continue

    merged = pd.concat(long_parts, ignore_index=True)

    merged = (
        merged
        .drop_duplicates(subset=["date", "variable", "scenario"], keep="first")
        .sort_values(["date", "scenario", "variable"])
        .reset_index(drop=True)
    )

    merged.insert(0, "sustax_total", f"{base_name}Total")
    merged.insert(1, "lat", lat)
    merged.insert(2, "lon", lon)

    out_name = f"{base_name}Total_LONG_mm.csv"
    out_fp = os.path.join(MERGED_LONG_DIR, out_name)
    merged.to_csv(out_fp, index=False)

    merged_rows.append({
        "sustax_total": f"{base_name}Total",
        "lat": lat,
        "lon": lon,
        "n_original_files": len(grp),
        "original_files": " | ".join(grp["sustax_file"].tolist()),
        "n_rows_merged": len(merged),
        "scenarios_merged": ", ".join(sorted(merged["scenario"].dropna().unique())),
        "out_file": out_fp
    })

merged_summary_df = pd.DataFrame(merged_rows).sort_values("sustax_total")
display(merged_summary_df.head())

merged_summary_path = os.path.join(OUT_DIR, "SUSTAX_TOTAL_LONG_summary.csv")
merged_summary_df.to_csv(merged_summary_path, index=False)

print("LONG totales guardados en:", MERGED_LONG_DIR)
print("Resumen LONG guardado en:", merged_summary_path)

summary_rows = []

long_files = sorted(glob.glob(os.path.join(MERGED_LONG_DIR, "*Total_LONG_mm.csv")))
print("Archivos LONG total detectados:", len(long_files))

for fp in long_files:
    fn = os.path.basename(fp)

    df = pd.read_csv(fp)

    if df.empty or "scenario" not in df.columns:
        continue

    sustax_total = df["sustax_total"].iloc[0]
    lat = df["lat"].iloc[0]
    lon = df["lon"].iloc[0]

    generated = []

    for sc in sorted(df["scenario"].dropna().unique()):
        sub = df[df["scenario"] == sc].copy()

        if sub.empty:
            continue

        piv = (
            sub.pivot_table(
                index="date",
                columns="variable",
                values="value_mm",
                aggfunc="mean"
            )
            .reset_index()
        )

        piv.insert(0, "sustax_total", sustax_total)
        piv.insert(1, "lat", lat)
        piv.insert(2, "lon", lon)

        out_name = f"{sustax_total}__{sc}__mm.csv"
        out_fp = os.path.join(BYSCEN_TOTAL_DIR, out_name)
        piv.to_csv(out_fp, index=False)

        generated.append(out_name)

    summary_rows.append({
        "sustax_total": sustax_total,
        "lat": lat,
        "lon": lon,
        "n_scenarios_generated": len(generated),
        "scenarios_generated": " | ".join(generated)
    })

byscen_summary_df = pd.DataFrame(summary_rows).sort_values("sustax_total")
display(byscen_summary_df.head())

byscen_summary_path = os.path.join(BYSCEN_TOTAL_DIR, "SUSTAX_TOTAL_BYSCEN_summary.csv")
byscen_summary_df.to_csv(byscen_summary_path, index=False)

print("Archivos por escenario guardados en:", BYSCEN_TOTAL_DIR)
print("Resumen guardado en:", byscen_summary_path)